# 1. Library Installations
Uses Colab's native Python 3.12 environment.

In [ ]:
%pip install torchao --upgrade
%pip install --upgrade transformers accelerate torchvision peft bitsandbytes pillow pandas tqdm num2words

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━

# 2. Upload and Extract Data
Upload your Images.zip and master_dataset.csv.

In [ ]:
from google.colab import files
print("Please upload Images.zip and master_dataset.csv")
uploaded = files.upload()
uploaded = files.upload()

Please upload Images.zip and master_dataset.csv


Saving master_dataset.csv to master_dataset.csv


Saving Images.zip to Images.zip


In [ ]:
import zipfile
import os

zip_path = "Images.zip"
extract_dir = "images"

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Extracted images:", len(os.listdir(extract_dir)))
else:
    print("Images.zip not found. Please ensure it was uploaded.")

Extracted images: 1


# 3. Prepare Dataset

In [ ]:
import pandas as pd

try:
    df = pd.read_csv("master_dataset.csv")

    # FIX: Added the extra 'Images/' to the path to match your extracted folder structure
    df["image_path"] = df.apply(
        lambda row: f"images/Images/{row['modulation']}/{row['filename']}" if 'filename' in row else f"images/Images/{row['modulation']}/{row['image_path'].split('/')[-1]}",
        axis=1
    )

    # Create splits
    train_df = df.iloc[:800]
    val_df = df.iloc[800:1000] # Kept small for faster evaluation
    print(f"Loaded {len(train_df)} train samples and {len(val_df)} validation samples.")
except FileNotFoundError:
    print("master_dataset.csv not found. Please upload it.")

Loaded 800 train samples and 200 validation samples.


# 4. SmolVLM Dataset & Collator
Contains the chat templates and -100 masking for correct Causal LM training.

In [ ]:
from torch.utils.data import Dataset
import torch
from PIL import Image

class VLMDataset(Dataset):
    def __init__(self, df, processor, is_eval=False):
        self.df = df
        self.processor = processor
        self.is_eval = is_eval

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')

        # 1. Calculate the true label strings
        impairments = []
        if row.get('phase_noise', 0) == 1: impairments.append('phase_noise')
        if row.get('iq_imbalance', 0) == 1: impairments.append('iq_imbalance')
        if row.get('jamming', 0) == 1: impairments.append('jamming')
        if row.get('amplitude_dist', 0) == 1: impairments.append('amplitude_dist')
        impairment = 'none' if not impairments else ('multiple' if len(impairments) > 1 else impairments[0])

        target_json = f'{{"modulation": "{row["modulation"]}", "impairment": "{impairment}", "snr": "{row["snr_level"]}"}}'

        # 2. Setup the Base Prompt
        messages = [
            {
                'role': 'user',
                'content': [
                    {'type': 'image'},
                    {'type': 'text', 'text': 'What is the modulation, impairment, and SNR?\nRespond strictly in valid JSON format like: {"modulation": "...", "impairment": "...", "snr": "..."}'}
                ]
            }
        ]

        if self.is_eval:
            # EVALUATION: Only give the prompt so the model has to guess the rest
            text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = self.processor(text=[text], images=[image], return_tensors='pt')

            item = {k: v.squeeze(0) for k, v in inputs.items()}
            item['raw_true_label'] = target_json
            return item

        else:
            # TRAINING: Append the true answer so the model can read it and learn it!
            messages.append({
                'role': 'assistant',
                'content': [{'type': 'text', 'text': target_json}]
            })

            # Generate the FULL sequence (Prompt + JSON Answer)
            text = self.processor.apply_chat_template(messages, tokenize=False)
            inputs = self.processor(text=[text], images=[image], return_tensors='pt')

            # Generate ONLY THE PROMPT to find exactly how many tokens long it is
            prompt_text = self.processor.apply_chat_template([messages[0]], tokenize=False, add_generation_prompt=True)
            prompt_inputs = self.processor(text=[prompt_text], images=[image], return_tensors='pt')
            prompt_len = prompt_inputs['input_ids'].shape[1]

            # Masking logic: Hide the prompt, leave the JSON answer visible to the loss function!
            labels = inputs['input_ids'].clone()
            labels[0, :prompt_len] = -100

            item = {k: v.squeeze(0) for k, v in inputs.items()}
            item['labels'] = labels.squeeze(0)
            return item

# 5. Initialize SmolVLM & DataLoaders

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model

model_id = 'HuggingFaceTB/SmolVLM2-500M-Video-Instruct'
processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map='auto'
)

# THE MAGIC BULLET: This slashes VRAM usage by ~70% during training!
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if 'train_df' in locals():
    train_dataset = VLMDataset(train_df, processor, is_eval=False)
    val_dataset = VLMDataset(val_df, processor, is_eval=True)
    # Keeping batch size at 1 for safety
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

W0602 10:59:42.953000 2633 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0602 10:59:43.013000 2633 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommende

processor_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.03G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

ERROR:bitsandbytes.cextension:bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file: No such file or directory


trainable params: 4,161,536 || all params: 511,643,840 || trainable%: 0.8134


# 6. Training and Evaluation Loop

In [ ]:
df["image_path"] = df.apply(
    lambda row: f"images/{row['filename']}" if 'filename' in row else f"images/{row['image_path'].split('/')[-1]}",
    axis=1
)

diagnostic evaluation format

In [ ]:
import random
import torch
import json
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel

print("Clearing corrupted memory hooks...")
# 1. Clear out memory caches completely
torch.cuda.empty_cache()

# 2. Reload a completely clean, unhooked copy of the base model
model_id = 'HuggingFaceTB/SmolVLM2-500M-Video-Instruct'
clean_base_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map='auto'
)

# 3. Pull a random verification sample from your validation dataset
sample_idx = random.randint(0, len(val_df) - 1)
row = val_df.iloc[sample_idx]

# Load target image
image = Image.open(row['image_path']).convert('RGB')

# Format the text using chat templates
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'image'},
            {'type': 'text', 'text': 'What is the modulation, impairment, and SNR?\nRespond strictly in valid JSON format like: {"modulation": "...", "impairment": "...", "snr": "..."}'}
        ]
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors='pt').to(clean_base_model.device)

# 4. Generate with the completely unhooked base model
with torch.no_grad():
    generated_ids = clean_base_model.generate(**inputs, max_new_tokens=50)

input_len = inputs['input_ids'].shape[1]
base_output = processor.batch_decode(generated_ids[:, input_len:], skip_special_tokens=True)[0]

print("\n--- BASE CLEAN MODEL OUTPUT (Before fine-tuning look) ---")
print(repr(base_output))

# Determine true fields
impairments = []
if row.get('phase_noise', 0) == 1: impairments.append('phase_noise')
if row.get('iq_imbalance', 0) == 1: impairments.append('iq_imbalance')
if row.get('jamming', 0) == 1: impairments.append('jamming')
if row.get('amplitude_dist', 0) == 1: impairments.append('amplitude_dist')
impairment = 'none' if not impairments else ('multiple' if len(impairments) > 1 else impairments[0])
true_label = f'{{"modulation": "{row["modulation"]}", "impairment": "{impairment}", "snr": "{row["snr_level"]}"}}'

print("\n--- EXPECTED TARGET LABEL ---")
print(repr(true_label))

Clearing corrupted memory hooks...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`



--- BASE CLEAN MODEL OUTPUT (Before fine-tuning look) ---
' {"modulation": "100%", "impairment": "100%", "snr": "100%"}'

--- EXPECTED TARGET LABEL ---
'{"modulation": "32-APSK", "impairment": "multiple", "snr": "High"}'


trainning

In [ ]:
import warnings
import os
warnings.filterwarnings("ignore")

from tqdm import tqdm
from torch.optim import AdamW

if 'train_loader' in locals():
    optimizer = AdamW(model.parameters(), lr=5e-5)

    # Set to 1 for a quick verification run
    epochs = 1

    print("Starting 1-Epoch verification training...")
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc=f'Epoch {epoch+1} Training'):
            optimizer.zero_grad()
            batch = {k: v.to(model.device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f'\nEpoch {epoch+1} Complete. Average Loss: {avg_loss:.4f}')

        # Save the checkpoint so your progress is locked in
        checkpoint_dir = f"./smolvlm_epoch_{epoch+1}"
        model.save_pretrained(checkpoint_dir)
        print(f"Checkpoint saved securely to {checkpoint_dir}")

    print("\nTraining complete! Proceed to the evaluation cell.")

Starting 1-Epoch verification training...


Epoch 1 Training:   0%|          | 0/800 [00:00<?, ?it/s][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Epoch 1 Training:   0%|          | 1/800 [00:04<58:05,  4.36s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Epoch 1 Training:   0%|          | 2/800 [00:12<1:29:29,  6.73s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Epoch 1 Training:   0%|          | 3/800 [00:17<1:16:38,  5.77s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs


Epoch 1 Complete. Average Loss: 0.0010
Checkpoint saved securely to ./smolvlm_epoch_1

Training complete! Proceed to the evaluation cell.


evaluation

In [ ]:
import json
import torch
from tqdm import tqdm

# 1. Disable checkpointing to unblock the generation head
model.gradient_checkpointing_disable()
model.eval()

# 2. Clear out the residual training memory from the GPU
torch.cuda.empty_cache()

correct = 0
total = 0

print("Running Post-Training Evaluation...")
with torch.no_grad():
    for batch in tqdm(val_loader, desc='Evaluating'):
        true_labels = batch.pop('raw_true_label', [])
        batch = {k: v.to(model.device) for k, v in batch.items()}

        # Generate tokens safely now that memory hooks are cleared
        outputs = model.generate(**batch, max_new_tokens=50)

        input_len = batch['input_ids'].shape[1]
        generated_tokens = outputs[:, input_len:]
        generated_texts = processor.batch_decode(generated_tokens, skip_special_tokens=True)

        for pred_text, true_text in zip(generated_texts, true_labels):
            try:
                # Clean up string wrapper issues (like Markdown code blocks)
                pred_cleaned = pred_text.strip().replace("```json", "").replace("```", "").strip()

                # Parse strings into true Python dictionaries
                pred_json = json.loads(pred_cleaned)
                true_json = json.loads(true_text.strip())

                # Compare semantic values rather than exact character spacing
                if pred_json == true_json:
                    correct += 1
            except Exception:
                # Fallback to strict string match if the model hallucinated invalid JSON syntax
                if pred_text.strip() == true_text.strip():
                    correct += 1
            total += 1

accuracy = (correct / total * 100) if total > 0 else 0
print(f'\nFinal Fine-Tuned Validation Accuracy: {accuracy:.2f}%')

Running Post-Training Evaluation...


Evaluating: 100%|██████████| 200/200 [04:10<00:00,  1.25s/it]


Final Fine-Tuned Validation Accuracy: 0.00%


trying trainning with evaluation

In [ ]:
from tqdm import tqdm
from torch.optim import AdamW

def evaluate_model(model, dataloader, processor):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            true_labels = batch.pop('raw_true_label', [])
            batch = {k: v.to(model.device) for k, v in batch.items()}

            outputs = model.generate(**batch, max_new_tokens=50)

            input_len = batch['input_ids'].shape[1]
            generated_tokens = outputs[:, input_len:]
            generated_texts = processor.batch_decode(generated_tokens, skip_special_tokens=True)

            for pred_text, true_text in zip(generated_texts, true_labels):
                if pred_text.strip() == true_text.strip():
                    correct += 1
                total += 1

    accuracy = (correct / total * 100) if total > 0 else 0
    print(f'\nValidation Accuracy: {accuracy:.2f}%\n')
    return accuracy

if 'train_loader' in locals():
    optimizer = AdamW(model.parameters(), lr=5e-5)
    epochs = 3

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f'Epoch {epoch+1} Training'):
            optimizer.zero_grad()
            batch = {k: v.to(model.device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f'\nEpoch {epoch+1} Complete. Average Loss: {avg_loss:.4f}')
        evaluate_model(model, val_loader, processor)

Epoch 1 Training:   0%|          | 0/800 [00:00<?, ?it/s][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Epoch 1 Training:   0%|          | 1/800 [00:05<1:16:32,  5.75s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Epoch 1 Training:   0%|          | 2/800 [00:09<1:03:57,  4.81s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Epoch 1 Training:   0%|          | 3/800 [00:


Epoch 1 Complete. Average Loss: 0.1050


Evaluating: 100%|██████████| 200/200 [04:28<00:00,  1.34s/it]



Validation Accuracy: 0.00%



Epoch 2 Training:   0%|          | 0/800 [00:00<?, ?it/s][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Epoch 2 Training:   0%|          | 1/800 [00:03<42:05,  3.16s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Epoch 2 Training:   0%|          | 2/800 [00:06<45:24,  3.41s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
Epoch 2 Training:   0%|          | 3/800 [00:09<43:51,  3.30s/it][transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` di

KeyboardInterrupt: 

trying more things hoping for a change (test)


In [ ]:
import random
import torch
import json
import gc
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel

print("Clearing VRAM of ghost hooks...")
# 1. Destroy the corrupted training model and clear the GPU cache
if 'model' in locals():
    del model
torch.cuda.empty_cache()
gc.collect()

print("Loading fresh base model...")
# 2. Load a pristine base model
model_id = 'HuggingFaceTB/SmolVLM2-500M-Video-Instruct'
processor = AutoProcessor.from_pretrained(model_id)
base_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map='auto'
)

print("Applying your Epoch 1 trained weights...")
# 3. Apply your trained adapter to the clean model
model = PeftModel.from_pretrained(base_model, "./smolvlm_epoch_1")
model.eval()

print("\n--- BEGIN DIAGNOSTIC ---\n")
# Test 3 random samples
with torch.no_grad():
    for i in range(3):
        sample_idx = random.randint(0, len(val_df) - 1)
        row = val_df.iloc[sample_idx]
        image = Image.open(row['image_path']).convert('RGB')

        messages = [
            {
                'role': 'user',
                'content': [
                    {'type': 'image'},
                    {'type': 'text', 'text': 'What is the modulation, impairment, and SNR?\nRespond strictly in valid JSON format like: {"modulation": "...", "impairment": "...", "snr": "..."}'}
                ]
            }
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors='pt').to(model.device)

        # Increased token limit to prevent truncation
        generated_ids = model.generate(**inputs, max_new_tokens=80)

        input_len = inputs['input_ids'].shape[1]
        pred_text = processor.batch_decode(generated_ids[:, input_len:], skip_special_tokens=True)[0]

        impairments = []
        if row.get('phase_noise', 0) == 1: impairments.append('phase_noise')
        if row.get('iq_imbalance', 0) == 1: impairments.append('iq_imbalance')
        if row.get('jamming', 0) == 1: impairments.append('jamming')
        if row.get('amplitude_dist', 0) == 1: impairments.append('amplitude_dist')
        impairment = 'none' if not impairments else ('multiple' if len(impairments) > 1 else impairments[0])
        true_label = f'{{"modulation": "{row["modulation"]}", "impairment": "{impairment}", "snr": "{row["snr_level"]}"}}'

        print(f"TEST {i+1}:")
        print(f"MODEL RAW OUTPUT : {repr(pred_text)}")
        print(f"EXPECTED TARGET  : {repr(true_label)}")
        print("-" * 50)

Clearing VRAM of ghost hooks...
Loading fresh base model...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Applying your Epoch 1 trained weights...


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`



--- BEGIN DIAGNOSTIC ---



[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


TEST 1:
MODEL RAW OUTPUT : ' '
EXPECTED TARGET  : '{"modulation": "32-APSK", "impairment": "multiple", "snr": "Medium"}'
--------------------------------------------------


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


TEST 2:
MODEL RAW OUTPUT : ' '
EXPECTED TARGET  : '{"modulation": "32-APSK", "impairment": "multiple", "snr": "Medium"}'
--------------------------------------------------
TEST 3:
MODEL RAW OUTPUT : ' '
EXPECTED TARGET  : '{"modulation": "32-APSK", "impairment": "none", "snr": "High"}'
--------------------------------------------------


at this point, i am almost sure that the problem is the prompt for the answer. The model seems to be trainning as expected but when it comes to evaluation it seems like it completely lost everything. Also, according to the evaluation testing block, the model raw output is " ", eventhough it should right something even without training/ fine tuning. However, due to limitations of google colab (I have made many google mails untill now) and limitations in my time, I will have to accept the 0%.

ή, οπως θα ελεγε (είπε ) το gemini:

Evaluation and Pipeline Limitations
During the evaluation phase, the fine-tuned model yielded a 0.00% validation accuracy. Diagnostic testing revealed that the model was outputting empty strings (" ") rather than incorrect classifications. Because even an untrained base language model will conditionally generate hallucinated text rather than blank space, this indicates a mechanical disconnect in the token generation pipeline rather than a failure of the model to learn weight representations.

The most probable cause is an error in the target masking function during data collation. If the -100 ignore index inadvertently masks the target JSON answer alongside the user prompt, the model minimizes its loss by learning to immediately output an End-Of-Sequence (EOS) token, resulting in a mute model. Due to strict constraints on Google Colab compute units and time limitations, further iterations to debug the dataset tokenization length and re-run the multi-hour training epochs were not feasible. Therefore, the 0% accuracy is accepted as a pipeline artifact rather than a true reflection of the model's theoretical capability.